# Model Build & Train
## Wide & Deep Career Recommender — Official Huawei MindSpore Implementation

**Prerequisite:** Notebook 10 must pass all checks before running this.

**Model source:** `src/` — vendored from `mindspore-ai/models` (commit `eab643f51`)
The official `WideDeepModel`, `NetWithLossClass`, `TrainStepWrap`, and
`PredictWithSigmoid` classes are imported directly from that directory.
We do not rewrite the model — only adapt our data to its expected format.

**Data format bridge:** The official model uses a unified sparse representation
`(feat_ids, feat_vals)` where every feature — categorical, continuous, and skill —
is encoded as a (global_ID, weight) pair. A `map()` transform converts our
MindRecord columns into this format before training begins.

**Steps:**
1. Configuration — imports, device, vocab offsets, model config
2. Data pipeline — feat_ids / feat_vals adapter and dataset creation
3. Official model — import, instantiate, parameter count, dry-run
4. Training setup — loss wrapper, TrainStepWrap, history
5. Training loop — epoch loop with wide & deep losses, live chart
6. Save artefacts — checkpoints, config, history
7. Evaluation — AUCMetric, ROC curve, confusion matrix

---

## Step 1: Configuration

In [ ]:
import os
import sys
import json
import pickle
import time
import shutil
import subprocess
import warnings
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 6)

# ── MindSpore ────────────────────────────────────────────────────────────────
import mindspore as ms
import mindspore.dataset as ds
import mindspore.common.dtype as mstype
import mindspore.ops as ops

# ── Add repo root to path (vendored model lives in <repo>/src) ───────────────
# Walk upward from the notebook until we find the vendored model source, so the
# notebook works from any working directory without depending on /workspace/.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "wide_and_deep.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wide_and_deep.py")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("✓ MindSpore  :", ms.__version__)
print("✓ NumPy      :", np.__version__)
print(f"  Started    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Repo root  : {REPO_ROOT}")

In [ ]:
# ── Device setup ─────────────────────────────────────────────────────────────
# Use ms.set_device() to avoid the deprecated device_target param in set_context.
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is not None:
    result = subprocess.run([nvidia_smi], capture_output=True)
    DEVICE_TARGET = "GPU" if result.returncode == 0 else "CPU"
else:
    DEVICE_TARGET = "CPU"

ms.set_context(mode=ms.GRAPH_MODE)   # mode param is not deprecated

try:
    ms.set_device(DEVICE_TARGET)
    print(f"✓ Device set via ms.set_device('{DEVICE_TARGET}')")
except AttributeError:
    # Fallback for older MindSpore builds
    from mindspore import context
    context.set_context(device_target=DEVICE_TARGET)
    print(f"✓ Device set via context.set_context (fallback) — '{DEVICE_TARGET}'")

if DEVICE_TARGET == "CPU":
    print()
    print("⚠  CPU mode active. Training will be slower than GPU.")
    print("   Recommended: batch_size=256, epochs=5 for a first run.")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
data_dir       = Path('/workspace/data/processed')
mindrecord_dir = Path('/workspace/data/mindrecord')
ckpt_dir       = Path('/workspace/checkpoints')
log_dir        = Path('/workspace/logs')
viz_dir        = Path('/workspace/data/visualizations')

for d in [ckpt_dir, log_dir]:
    d.mkdir(parents=True, exist_ok=True)

MINDRECORD_FILE = str(mindrecord_dir / 'widedeep_training.mindrecord')

print("✓ Paths configured")
print(f"  MindRecord : {MINDRECORD_FILE}")
print(f"  Checkpoints: {ckpt_dir}")
print(f"  Logs       : {log_dir}")

In [ ]:
print("="*80)
print("COMPUTING VOCABULARY OFFSETS FROM LABEL ENCODERS")
print("="*80)

# Load encoders to get the true vocab size of each categorical feature.
# The official model uses a single embedding table for ALL features,
# so we need global IDs: each categorical feature's IDs are offset by the
# cumulative sum of all previous features' vocab sizes.

with open(data_dir / 'label_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

print(f"\n✓ Loaded {len(encoders)} encoder(s)")

# ── Map encoder key → MindRecord column name ──────────────────────────────────
# Adjust this mapping if your encoder keys differ from the column names.
ENCODER_KEY_MAP = {
    'education_level'              : 'education_level_encoded',
    'field_of_study'               : 'field_of_study_encoded',
    'simulation_categoria'         : 'simulation_categoria_encoded',
    'simulation_nivel_dificultad'  : 'simulation_nivel_dificultad_encoded',
    'simulation_industria'         : 'simulation_industria_encoded',
}

CAT_FEATURE_ORDER = list(ENCODER_KEY_MAP.keys())

def get_vocab_size(enc):
    if hasattr(enc, 'classes_'):
        return len(enc.classes_)
    elif isinstance(enc, dict):
        return len(enc)
    return 50   # safe fallback

CAT_VOCAB_SIZES = []
for enc_key in CAT_FEATURE_ORDER:
    enc = encoders.get(enc_key)
    if enc is None:
        # Try partial match
        matches = [v for k, v in encoders.items() if enc_key.split('_')[0] in k]
        enc = matches[0] if matches else None
    vocab = get_vocab_size(enc) if enc is not None else 50
    CAT_VOCAB_SIZES.append(vocab)

# Cumulative offsets: field i starts at CAT_OFFSETS[i] in the global vocab
CAT_OFFSETS    = np.cumsum([0] + CAT_VOCAB_SIZES[:-1], dtype=np.int32)
CONT_BASE_ID   = int(sum(CAT_VOCAB_SIZES))      # first ID reserved for continuous fields
SKILL_BASE_ID  = CONT_BASE_ID + 6               # 6 continuous features
SIM_SKILL_BASE = SKILL_BASE_ID + 52             # 52 user skill slots
VOCAB_SIZE     = SIM_SKILL_BASE + 52            # 52 sim skill slots

# field_size = 5 categorical + 6 continuous + 52 user skills + 52 sim skills
FIELD_SIZE = 5 + 6 + 52 + 52   # = 115

print(f"\nCategorical feature vocab sizes:")
for name, vocab, offset in zip(CAT_FEATURE_ORDER, CAT_VOCAB_SIZES, CAT_OFFSETS):
    print(f"  {name:<40}: vocab={vocab:>5}  global_offset={offset}")

print(f"\nID ranges in global vocabulary:")
print(f"  Categorical IDs  : 0  – {CONT_BASE_ID - 1}")
print(f"  Continuous IDs   : {CONT_BASE_ID} – {SKILL_BASE_ID - 1}  (one fixed ID per field)")
print(f"  User skill IDs   : {SKILL_BASE_ID} – {SIM_SKILL_BASE - 1}")
print(f"  Sim  skill IDs   : {SIM_SKILL_BASE} – {VOCAB_SIZE - 1}")
print(f"\n  Total vocab_size : {VOCAB_SIZE}")
print(f"  field_size       : {FIELD_SIZE}")

In [ ]:
print("="*80)
print("HYPERPARAMETER CONFIGURATION")
print("="*80)

# All attributes accessed by WideDeepModel.__init__ and NetWithLossClass
# must be present in config. SimpleNamespace lets us set them like object
# attributes (config.batch_size, config.field_size, etc.)

BATCH_SIZE = 256
# ── Epoch budget ─────────────────────────────────────────────────────────────
# With early stopping (patience=40), training stops automatically when AUC
# stops improving. Set this high; the stopping rule caps actual epochs used.
EPOCHS     = 1000
RUN_NAME   = 'baseline'

# ── Learning rates for DAOTrainStepWrap ──────────────────────────────────────
# Root cause of the 0.72 plateau: the official FTRL lr=5e-2 was calibrated for
# 45M samples (default_config.yaml). At 18k samples, each epoch sees the full
# dataset ~57× more often, so the effective learning rate is 57× too high.
# Scaled down by sqrt(18k / 45M) ≈ 0.020 → FTRL 5e-2 × 0.020 ≈ 1e-3.
# Adam is less sensitive but also benefits from reduction on small data.
WIDE_LR = 1e-3    # kept for reference — was the wide LR target (FTRL deprecated in 2.8.0)
                   # TrainOneStepCell uses a single DEEP_LR for all params
DEEP_LR = 1e-4    # Adam (deep parameters)  — was 3.5e-4

config = SimpleNamespace(
    # ── Dimensions ─────────────────────────────────────────────────────────
    batch_size       = BATCH_SIZE,
    eval_batch_size  = BATCH_SIZE,
    field_size       = FIELD_SIZE,        # 115
    vocab_size       = VOCAB_SIZE,        # computed from encoders

    # emb_dim=16: the official default of 80 is for 200k-vocab, 45M-sample data.
    # At 18k samples with a small vocab this is scaled way down. emb_dim=16 is
    # the value actually used for the reference checkpoint
    # (checkpoints/baseline/dao_wide_deep_best.ckpt): embedding_table shape
    # (148, 16), dense_layer_1 shape (1840, 256) = field_size 115 × emb_dim 16.
    # Do NOT lower this to 8 — the checkpoint will not load.
    emb_dim          = 16,

    # ── Deep network ────────────────────────────────────────────────────────
    # MUST have exactly 4 elements — WideDeepModel always creates 5 DenseLayers
    deep_layer_dim   = [256, 128, 64, 32],  # matches dao_wide_deep_best.ckpt
    deep_layer_act   = 'relu',
    dropout_flag     = True,              # dropout combats overfitting
    keep_prob        = 0.7,              # 30% dropout; DenseLayer uses p=1-keep_prob

    # ── Regularisation ──────────────────────────────────────────────────────
    # l2_coef=1e-3: 12× stronger than default 8e-5, appropriate for small data
    l2_coef          = 1e-3,

    # ── Initialisation ──────────────────────────────────────────────────────
    weight_bias_init = ['normal', 'normal'],
    emb_init         = 'normal',
    init_args        = [-0.01, 0.01],

    # ── Distribution / server (all disabled for standalone CPU) ─────────────
    host_device_mix   = 0,
    parameter_server  = 0,
    sparse            = False,
    field_slice       = False,
    full_batch        = False,
    manual_shape      = None,
    vocab_cache_size  = 0,
    deep_table_slice_mode = 'column_slice',
    use_sp            = True,

    # ── Logging ─────────────────────────────────────────────────────────────
    loss_file_name   = str(log_dir / f'loss_{RUN_NAME}.log'),
    eval_file_name   = str(log_dir / f'eval_{RUN_NAME}.log'),
    stra_ckpt        = str(ckpt_dir / 'strategy.ckpt'),
)

print()
print(f"  {'batch_size':<25}: {config.batch_size}")
print(f"  {'field_size':<25}: {config.field_size}")
print(f"  {'vocab_size':<25}: {config.vocab_size}")
print(f"  {'emb_dim':<25}: {config.emb_dim}  (matches dao_wide_deep_best.ckpt)")
print(f"  {'deep_input_dims':<25}: {config.field_size * config.emb_dim}  (field_size × emb_dim)")
print(f"  {'deep_layer_dim':<25}: {config.deep_layer_dim}")
print(f"  {'deep_layer_act':<25}: {config.deep_layer_act}")
print(f"  {'dropout_flag':<25}: {config.dropout_flag}  keep_prob={config.keep_prob}")
print(f"  {'l2_coef':<25}: {config.l2_coef}  (was 8e-5 → stronger regularisation)")
print(f"  {'adam_lr (all params)':<25}: {DEEP_LR}  (single Adam, TrainOneStepCell)")
print(f"  {'epochs (budget)':<25}: {EPOCHS}  (early stopping will terminate earlier)")
print(f"  {'device_target':<25}: {DEVICE_TARGET}")

---
## Step 2: Data Pipeline

### Why (feat_ids, feat_vals)?

The official `WideDeepModel` uses a single embedding table for every feature.
Each sample must be represented as `field_size` (ID, weight) pairs:

| Feature type | ID | Weight |
|---|---|---|
| Categorical (e.g. education) | `encoded_value + vocab_offset` | `1.0` |
| Continuous (e.g. analytical_score) | fixed field marker ID | `normalised_value` |
| Skill vector (52-dim multi-hot) | `skill_base_id + position` | `0.0 or 1.0` |

Our 115 fields: **5 categorical + 6 continuous + 52 user skills + 52 sim skills**

The `map()` transform below converts our 14 MindRecord columns into the three
tensors the official model expects: `feat_ids [115]`, `feat_vals [115]`, `label [1]`.

In [ ]:
# ── Columns to load from MindRecord ──────────────────────────────────────────
LOAD_COLS = [
    'education_level_encoded',
    'field_of_study_encoded',
    'simulation_categoria_encoded',
    'simulation_nivel_dificultad_encoded',
    'simulation_industria_encoded',
    'analytical_score',
    'creative_score',
    'social_score',
    'linguistic_score',
    'hands_on_score',
    'simulation_duracion_horas',
    'user_skills',
    'simulation_skills',
    'label',
]

# Pre-build the fixed ID arrays so they are not recomputed every sample
_CONT_IDS       = np.arange(CONT_BASE_ID, CONT_BASE_ID + 6,  dtype=np.int32)
_USER_SKILL_IDS = np.arange(SKILL_BASE_ID, SKILL_BASE_ID + 52, dtype=np.int32)
_SIM_SKILL_IDS  = np.arange(SIM_SKILL_BASE, SIM_SKILL_BASE + 52, dtype=np.int32)
_CAT_OFFSETS    = CAT_OFFSETS   # (5,) int32


def build_widedeep_features(edu, fos, cat_feat, lvl, ind,
                              anal, crea, soc, ling, hand, dur,
                              user_sk, sim_sk, label_val):
    """
    Map function: converts 14 MindRecord columns → feat_ids, feat_vals, label.

    Called once per sample by MindDataset.map().
    All inputs and outputs are numpy arrays.
    """
    # ── Feature IDs ───────────────────────────────────────────────────────────
    cat_ids = np.array([
        int(edu)      + int(_CAT_OFFSETS[0]),
        int(fos)      + int(_CAT_OFFSETS[1]),
        int(cat_feat) + int(_CAT_OFFSETS[2]),
        int(lvl)      + int(_CAT_OFFSETS[3]),
        int(ind)      + int(_CAT_OFFSETS[4]),
    ], dtype=np.int32)

    feat_ids = np.concatenate([cat_ids, _CONT_IDS, _USER_SKILL_IDS, _SIM_SKILL_IDS])

    # ── Feature weights ───────────────────────────────────────────────────────
    cat_vals  = np.ones(5, dtype=np.float32)
    cont_vals = np.array([
        float(anal) / 100.0,
        float(crea) / 100.0,
        float(soc)  / 100.0,
        float(ling) / 100.0,
        float(hand) / 100.0,
        float(dur)  / 20.0,
    ], dtype=np.float32)

    feat_vals = np.concatenate([
        cat_vals,
        cont_vals,
        user_sk.flatten().astype(np.float32),
        sim_sk.flatten().astype(np.float32),
    ])

    # ── Label: float32 [1] for SigmoidCrossEntropyWithLogits ─────────────────
    lbl = np.array([float(label_val)], dtype=np.float32)

    return feat_ids, feat_vals, lbl

print("✓ build_widedeep_features map function defined")
print(f"  Output feat_ids shape  : ({FIELD_SIZE},) int32")
print(f"  Output feat_vals shape : ({FIELD_SIZE},) float32")
print(f"  Output label shape     : (1,) float32")

In [ ]:
def create_split_dataset(mindrecord_file, n_skip=0, n_take=None,
                          batch_size=256, shuffle_buffer=0,
                          drop_remainder=True, num_parallel_workers=4):
    """
    Creates a correctly partitioned MindDataset.

    The key design principle:
      - MindDataset is ALWAYS opened with shuffle=False here.
      - skip/take operate on the same sequential ordering for both train
        and test, guaranteeing a clean non-overlapping partition.
      - shuffle_buffer > 0 adds a pipeline-level shuffle AFTER splitting,
        so only the train set is shuffled without disturbing the split boundary.

    drop_remainder=True is required because WideDeepModel.construct() uses
    self.batch_size as a fixed integer in a Reshape op.
    """
    dataset = ds.MindDataset(
        dataset_files=mindrecord_file,
        columns_list=LOAD_COLS,
        shuffle=False,              # always False — split first, shuffle after
        num_parallel_workers=num_parallel_workers
    )
    if n_skip > 0:
        dataset = dataset.skip(n_skip)
    if n_take is not None:
        dataset = dataset.take(n_take)
    dataset = dataset.map(
        operations=build_widedeep_features,
        input_columns=LOAD_COLS,
        output_columns=['feat_ids', 'feat_vals', 'label']
    )
    if shuffle_buffer > 0:
        dataset = dataset.shuffle(buffer_size=shuffle_buffer)
    dataset = dataset.batch(batch_size, drop_remainder=drop_remainder)
    return dataset

print("✓ create_split_dataset factory defined")
print("  shuffle=False on MindDataset → skip/take on same ordering → no overlap")

In [ ]:
print("="*80)
print("CREATING TRAIN / TEST DATASETS  (sequential partition)")
print("="*80)

# Count total samples from the unshuffled file
size_ds = ds.MindDataset(MINDRECORD_FILE, columns_list=['label'], shuffle=False)
TOTAL_SAMPLES = size_ds.get_dataset_size()
TRAIN_N       = int(TOTAL_SAMPLES * 0.8)   # sample-level split boundary
TEST_N        = TOTAL_SAMPLES - TRAIN_N

print(f"\nTotal samples   : {TOTAL_SAMPLES:,}")
print(f"Train samples   : {TRAIN_N:,}  (first 80% of file — sequential)")
print(f"Test  samples   : {TEST_N:,}   (last  20% of file — sequential)")
print()
print("Split method: skip/take on unshuffled source → no overlap guaranteed")
print("Train shuffle: pipeline .shuffle() after split, not before")

# Train: first TRAIN_N samples → shuffle as pipeline step → batch
# Test:  last TEST_N  samples → no shuffle                 → batch
SHUFFLE_BUFFER = min(10000, TRAIN_N)

train_dataset = create_split_dataset(
    MINDRECORD_FILE,
    n_skip=0, n_take=TRAIN_N,
    batch_size=BATCH_SIZE,
    shuffle_buffer=SHUFFLE_BUFFER,
    drop_remainder=True
)

test_dataset = create_split_dataset(
    MINDRECORD_FILE,
    n_skip=TRAIN_N, n_take=None,
    batch_size=BATCH_SIZE,
    shuffle_buffer=0,
    drop_remainder=True
)

STEPS_PER_EPOCH = train_dataset.get_dataset_size()
TEST_STEPS      = test_dataset.get_dataset_size()

print(f"\nTrain batches   : {STEPS_PER_EPOCH:,}  (batch_size={BATCH_SIZE}, drop_remainder=True)")
print(f"Test  batches   : {TEST_STEPS:,}")
print(f"Total steps     : {STEPS_PER_EPOCH * EPOCHS:,}  ({EPOCHS} epochs)")

# Sanity check: confirm no sample can appear in both splits
# (guaranteed by construction since both skip from the same unshuffled file)
print()
print("✓ Split integrity: train=[0, TRAIN_N)  test=[TRAIN_N, TOTAL_SAMPLES)")
print("  No sample can appear in both sets.")

In [ ]:
print("="*80)
print("CLASS DISTRIBUTION & POS_WEIGHT")
print("="*80)

# Scan the training split (unshuffled, same n_take=TRAIN_N) to count pos/neg.
# POS_WEIGHT = neg_count / pos_count scales positive gradients to correct
# for imbalance. A 4:1 imbalance → POS_WEIGHT=4.0 means positives get 4×
# the gradient signal per sample, preventing the model from ignoring them.

scan_ds = create_split_dataset(
    MINDRECORD_FILE, n_skip=0, n_take=TRAIN_N,
    batch_size=BATCH_SIZE, shuffle_buffer=0, drop_remainder=True
)
neg_count = 0
pos_count = 0
for batch in scan_ds.create_dict_iterator(output_numpy=True):
    lbl = batch['label'].flatten()
    pos_count += int((lbl == 1).sum())
    neg_count += int((lbl == 0).sum())

total_scanned = pos_count + neg_count
POS_WEIGHT    = neg_count / max(pos_count, 1)

print(f"\n  Positive samples : {pos_count:>8,}  ({100*pos_count/total_scanned:.1f}%)")
print(f"  Negative samples : {neg_count:>8,}  ({100*neg_count/total_scanned:.1f}%)")
print(f"  POS_WEIGHT       : {POS_WEIGHT:.4f}  (neg:pos ratio)")

if POS_WEIGHT >= 3.0:
    print(f"\n  ⚠  Severe imbalance ({POS_WEIGHT:.1f}×). Weighted loss is critical.")
elif POS_WEIGHT >= 1.5:
    print(f"\n  Moderate imbalance ({POS_WEIGHT:.1f}×). Weighted loss will help.")
else:
    print(f"\n  ✓ Near-balanced ({POS_WEIGHT:.1f}×). Weighted loss has minor effect.")

In [ ]:
print("="*80)
print("BATCH SHAPE VERIFICATION")
print("="*80)

inspect_iter = train_dataset.create_dict_iterator(output_numpy=True)
batch = next(inspect_iter)
del inspect_iter

feat_ids  = batch['feat_ids']
feat_vals = batch['feat_vals']
label     = batch['label']

print(f"\n{'Column':<15} {'Shape':<22} {'Dtype':<10} {'Min':>10} {'Max':>10}")
print("-"*70)
print(f"{'feat_ids':<15} {str(feat_ids.shape):<22} {str(feat_ids.dtype):<10}"
      f" {feat_ids.min():>10} {feat_ids.max():>10}")
print(f"{'feat_vals':<15} {str(feat_vals.shape):<22} {str(feat_vals.dtype):<10}"
      f" {feat_vals.min():>10.4f} {feat_vals.max():>10.4f}")
print(f"{'label':<15} {str(label.shape):<22} {str(label.dtype):<10}"
      f" {label.min():>10.1f} {label.max():>10.1f}")

assert feat_ids.shape  == (BATCH_SIZE, FIELD_SIZE),     f"feat_ids shape mismatch: {feat_ids.shape} vs ({BATCH_SIZE}, {FIELD_SIZE})"
assert feat_vals.shape == (BATCH_SIZE, FIELD_SIZE),     f"feat_vals shape mismatch: {feat_vals.shape} vs ({BATCH_SIZE}, {FIELD_SIZE})"
assert label.shape     == (BATCH_SIZE, 1),     f"label shape mismatch: {label.shape} vs ({BATCH_SIZE}, 1)"
assert feat_ids.max()  < VOCAB_SIZE,     f"feat_id {feat_ids.max()} exceeds vocab_size {VOCAB_SIZE}"

print("\n✓ All shapes and ranges correct — data pipeline is ready")

---
## Step 3: Official Model Setup

Imports and instantiates the official Huawei classes directly.
No model code is rewritten here — the vendored source files in this repo's
`src/` directory are used as-is.

**Class responsibilities:**

| Class | Role |
|---|---|
| `WideDeepModel` | Core network — wide embedding + deep DNN |
| `NetWithLossClass` | Adds `SigmoidCrossEntropyWithLogits` + L2 loss |
| `TrainStepWrap` | Dual FTRL (wide) + Adam (deep) with gradient scaling |
| `PredictWithSigmoid` | Inference wrapper — applies sigmoid to logits |
| `AUCMetric` | sklearn ROC-AUC accumulator |

In [ ]:
print("="*80)
print("IMPORTING OFFICIAL WIDE & DEEP CLASSES")
print("="*80)

from src.wide_and_deep import (
    WideDeepModel,
    NetWithLossClass,
    TrainStepWrap,
    PredictWithSigmoid,
)
from src.metrics import AUCMetric

print("\n✓ WideDeepModel      — core wide+deep network")
print("✓ NetWithLossClass    — SigmoidCrossEntropyWithLogits + L2")
print("✓ TrainStepWrap       — FTRL (wide) + Adam (deep) dual optimiser")
print("✓ PredictWithSigmoid  — inference with sigmoid activation")
print("✓ AUCMetric           — sklearn AUC-ROC accumulator")

In [ ]:
print("="*80)
print("EmbeddingLookup COMPATIBILITY SHIM")
print("="*80)

# The official wide_and_deep.py passes vocab_cache_size= to nn.EmbeddingLookup.
# Newer MindSpore versions removed that parameter. The shim below patches
# EmbeddingLookup.__init__ to silently drop it, so the official source file
# works unchanged on any MindSpore version.

import inspect
import mindspore.nn as _nn

_original_emb_init = _nn.EmbeddingLookup.__init__

_emb_params = set(inspect.signature(_original_emb_init).parameters.keys())

if 'vocab_cache_size' not in _emb_params:
    def _patched_emb_init(self, *args, **kwargs):
        kwargs.pop('vocab_cache_size', None)
        _original_emb_init(self, *args, **kwargs)
    _nn.EmbeddingLookup.__init__ = _patched_emb_init
    print("✓ Shim applied: vocab_cache_size stripped from EmbeddingLookup.__init__")
    print(f"  (MindSpore {ms.__version__} does not support this parameter)")
else:
    print("✓ No shim needed: this MindSpore version supports vocab_cache_size")

In [ ]:
print("="*80)
print("WeightedNetWithLossClass (CLASS-IMBALANCE CORRECTION)")
print("="*80)

class WeightedNetWithLossClass(ms.nn.Cell):
    """
    Replaces the official NetWithLossClass with a pos_weight-aware version.

    The official loss treats all samples equally. With ~82% negatives, the
    model learns to ignore positives (AUC plateaus at ~0.72).

    Fix: for each sample, scale its loss by:
        weight = 1 + (pos_weight - 1) * label
    which equals 1.0 for negatives and pos_weight for positives.

    Uses the same ops.SigmoidCrossEntropyWithLogits for numerical stability.
    auto_prefix=False mirrors the official class so parameter names match.
    """
    def __init__(self, network, config, pos_weight=1.0):
        super(WeightedNetWithLossClass, self).__init__(auto_prefix=False)
        self.network    = network
        self.l2_coef    = config.l2_coef
        self.loss       = ops.SigmoidCrossEntropyWithLogits()
        self.pw         = ms.Tensor([pos_weight], mstype.float32)
        self.square     = ops.Square()
        self.reduceMean = ops.ReduceMean(keep_dims=False)
        self.reduceSum  = ops.ReduceSum(keep_dims=False)

    def construct(self, batch_ids, batch_wts, label):
        predict, embedding_table = self.network(batch_ids, batch_wts)
        per_loss  = self.loss(predict, label)
        weight    = 1.0 + (self.pw - 1.0) * label   # 1 for neg, pos_weight for pos
        wide_loss = self.reduceMean(per_loss * weight)
        l2_loss_v = self.reduceSum(self.square(embedding_table)) / 2
        deep_loss = wide_loss + self.l2_coef * l2_loss_v
        return wide_loss, deep_loss

print(f"✓ WeightedNetWithLossClass defined")
print(f"  pos_weight = {POS_WEIGHT:.4f}  →  positive samples get {POS_WEIGHT:.1f}× the gradient signal")

In [ ]:
print("="*80)
print("INSTANTIATING MODEL")
print("="*80)

# WideDeepModel reads all its hyperparameters from config
network = WideDeepModel(config)

# WeightedNetWithLossClass: pos_weight corrects class imbalance
# (auto_prefix=False matches the official NetWithLossClass behaviour)
loss_net = WeightedNetWithLossClass(network, config, pos_weight=POS_WEIGHT)

print(f"\n  deep_input_dims    : {network.deep_input_dims}  (field_size × emb_dim)")
print(f"  all_dim_list       : {network.all_dim_list}")
print(f"  vocab_size         : {config.vocab_size}")
print(f"  field_size         : {config.field_size}")
print(f"  emb_dim            : {config.emb_dim}")
print(f"  pos_weight         : {POS_WEIGHT:.4f}")
print()
print("✓ WideDeepModel instantiated")
print("✓ WeightedNetWithLossClass instantiated (pos_weight applied)")

In [ ]:
print("="*80)
print("MODEL PARAMETER COUNT")
print("="*80)

def count_params(net):
    return sum(p.asnumpy().size for p in net.trainable_params())

# Parameter breakdown
wide_emb_params  = network.wide_embeddinglookup.embedding_table.asnumpy().size
deep_emb_params  = network.deep_embeddinglookup.embedding_table.asnumpy().size
deep_layer_params = sum(
    count_params(layer)
    for layer in [network.dense_layer_1, network.dense_layer_2,
                  network.dense_layer_3, network.dense_layer_4,
                  network.dense_layer_5]
)
wide_b_params  = network.wide_b.asnumpy().size
total_params   = count_params(network)

print(f"\n  Wide embedding table : {wide_emb_params:>12,}")
print(f"  Deep embedding table : {deep_emb_params:>12,}")
print(f"  Deep DNN layers      : {deep_layer_params:>12,}")
print(f"  Wide bias (wide_b)   : {wide_b_params:>12,}")
print(f"  {'─'*30}")
print(f"  TOTAL                : {total_params:>12,}")
print(f"  Memory (fp32)        : {total_params * 4 / 1024 / 1024:>10.1f} MB")

In [ ]:
print("="*80)
print("FORWARD PASS DRY-RUN")
print("="*80)

# Use a real batch from train_dataset so shapes are guaranteed correct
network.set_train(False)

dry_iter = train_dataset.create_dict_iterator()
dry_batch = next(dry_iter)
del dry_iter

out_logits, out_emb = network(dry_batch['feat_ids'], dry_batch['feat_vals'])

print(f"\n  Input  feat_ids  : {dry_batch['feat_ids'].shape}  int32")
print(f"  Input  feat_vals : {dry_batch['feat_vals'].shape}  float32")
print(f"  Output logits    : {out_logits.shape}   float32  (raw, no sigmoid)")
print(f"  Output emb_table : {out_emb.shape}  (embedding table for L2)")

assert out_logits.shape == (BATCH_SIZE, 1),     f"Logit shape mismatch: {out_logits.shape} vs ({BATCH_SIZE}, 1)"

print("\n✓ Forward pass correct — model is ready for training")

---
## Step 4: Training Setup

Uses `ms.nn.TrainOneStepCell` — MindSpore's standard, fully-tested training
wrapper. All custom training cells were replaced because:

- **`TrainStepWrap`**: FTRL lr hardcoded for 45M samples; FTRL uses the
  deprecated `SparseApplyFtrl` op (removed in MindSpore 2.8.0)
- **Custom `DAOTrainStepWrap` with `param_groups`**: triggers `is_group_lr=True`
  in Adam, which constructs per-parameter lr tensors shaped `[n,1]` — these
  must match the embedding table's moment buffer shape `[n, emb_dim]`, causing
  a `ValueError` at graph compilation
- **Custom `DAOTrainStepWrap` with two Adam instances + `GradOperation`**:
  `sens_param=True` requires a matching sensitivity tensor tuple for the two
  loss outputs, causing shape mismatches at the C++ level

`TrainOneStepCell(net, optimizer)` requires the wrapped network to return a
**scalar** loss. `_CombinedLossCell` provides that by summing `loss_w + loss_d`
from `WeightedNetWithLossClass`, preserving both the BCE signal and L2
regularisation in a single value.

The per-epoch `(loss_w, loss_d)` breakdown is recovered with a separate
forward-only call on one batch — negligible cost, clean separation.

In [ ]:
print("="*80)
print("BUILDING TRAINING STEP  (TrainOneStepCell + Adam)")
print("="*80)

class _CombinedLossCell(ms.nn.Cell):
    """
    Thin wrapper that returns loss_w + loss_d as a scalar.

    TrainOneStepCell requires its wrapped network to return a single scalar.
    WeightedNetWithLossClass returns (wide_loss, deep_loss). This cell sums
    them so the gradient flows through both BCE and L2 regularisation terms.

    auto_prefix=False: parameters are registered under the original names
    from WeightedNetWithLossClass, not re-prefixed with '_CombinedLossCell.'.
    """
    def __init__(self, loss_net):
        super(_CombinedLossCell, self).__init__(auto_prefix=False)
        self.net = loss_net

    def construct(self, batch_ids, batch_wts, label):
        loss_w, loss_d = self.net(batch_ids, batch_wts, label)
        return loss_w + loss_d

# Single Adam — scalar learning_rate, no param_groups, no is_group_lr issues.
# Using DEEP_LR (1e-4) for all parameters. On 18k samples the wide/deep LR
# split is a micro-optimisation that is not worth the implementation risk.
combined_cell = _CombinedLossCell(loss_net)
optimizer     = ms.nn.Adam(
    combined_cell.trainable_params(),
    learning_rate = DEEP_LR,
    eps           = 1e-8,
)
train_net = ms.nn.TrainOneStepCell(combined_cell, optimizer)
train_net.set_train()

n_params = sum(p.asnumpy().size for p in combined_cell.trainable_params())
print(f"\n  Total trainable params : {n_params:,}")
print(f"  Optimizer              : Adam  lr={DEEP_LR}  eps=1e-8")
print(f"  Loss cell              : _CombinedLossCell  (loss_w + loss_d)")
print()
print("✓ TrainOneStepCell ready")

In [ ]:
# Training history and output directories
RUN_CKPT = ckpt_dir / RUN_NAME
RUN_CKPT.mkdir(parents=True, exist_ok=True)

history = {
    'epoch'        : [],
    'total_loss'   : [],   # loss_w + loss_d (what TrainOneStepCell minimises)
    'auc'          : [],
    'epoch_time_s' : [],
}

print(f"✓ Run name       : {RUN_NAME}")
print(f"  Checkpoint dir : {RUN_CKPT}")

In [ ]:
from IPython.display import clear_output

def plot_history(history, save_path=None):
    if not history['epoch']:
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # ── Training loss ────────────────────────────────────────────────────────
    axes[0].plot(history['epoch'], history['total_loss'],
                 '-', color='#3498db', linewidth=1.5, label='Total loss (BCE + L2)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss'); axes[0].legend()

    # ── AUC: raw + smoothed ──────────────────────────────────────────────────
    auc_vals = [v for v in history['auc'] if v is not None]
    if auc_vals:
        auc_epochs = [e for e, v in zip(history['epoch'], history['auc']) if v is not None]
        # Raw
        axes[1].plot(auc_epochs, auc_vals,
                     '-', color='#2ecc71', linewidth=1, alpha=0.45, label='AUC raw')
        # Smoothed (5-epoch trailing mean)
        smooth = []
        for i in range(len(auc_vals)):
            window = auc_vals[max(0, i-4):i+1]
            smooth.append(float(np.mean(window)))
        axes[1].plot(auc_epochs, smooth,
                     '-', color='#27ae60', linewidth=2.5, label='AUC smoothed (5ep)')
        axes[1].axhline(0.75, color='orange', linestyle='--', linewidth=1.5,
                        label='AUC=0.75 target')
        best_idx = int(np.argmax(auc_vals))
        axes[1].axvline(auc_epochs[best_idx], color='red', linestyle=':', linewidth=1,
                        label=f'Best={max(auc_vals):.4f} ep{auc_epochs[best_idx]}')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC-ROC')
        axes[1].set_title('Test AUC-ROC per Epoch')
        axes[1].set_ylim(0.4, 1.0); axes[1].legend(fontsize=8)

    # ── Epoch time ───────────────────────────────────────────────────────────
    axes[2].bar(history['epoch'], [t / 60 for t in history['epoch_time_s']],
                color='#9b59b6', alpha=0.8)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Minutes')
    axes[2].set_title('Epoch Duration')

    ep_done = history['epoch'][-1]
    plt.suptitle(
        f"Training Progress — {RUN_NAME}  |  Epoch {ep_done}/{EPOCHS}  |  device={DEVICE_TARGET}",
        fontsize=12
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print("✓ plot_history defined (raw + smoothed AUC, best-epoch marker)")

---
## Step 5: Training Loop

Each call to `train_net(feat_ids, feat_vals, label)` performs:
1. Forward pass through `NetWithLossClass` → `(wide_loss, deep_loss)`
2. Separate backward passes: FTRL gradients for wide, Adam gradients for deep
3. Weight updates for both parameter groups simultaneously

**Healthy training signs:**
- Both `wide_loss` and `deep_loss` decrease across epochs
- AUC-ROC rises above 0.70 by epoch 3–5
- Losses do not diverge or become NaN

**Signs of trouble:**
- NaN loss → check feat_vals for Inf (re-run Notebook 10 Step 5)
- Flat AUC after epoch 3 → try larger `emb_dim` (32 or 64)
- wide_loss and deep_loss diverge significantly → expected: they use different optimisers

In [ ]:
def quick_auc_eval(eval_ds, net, auc_metric):
    """
    Run inference on eval_ds and return AUC-ROC.

    Uses the WideDeepModel directly (not PredictWithSigmoid) to avoid
    parameter-name issues from double-wrapping. Applies sigmoid manually.
    """
    net.set_train(False)
    auc_metric.clear()
    sigmoid_op = ops.Sigmoid()

    for data in eval_ds.create_dict_iterator():
        logits, _ = net(data['feat_ids'], data['feat_vals'])
        pred_probs = sigmoid_op(logits)
        auc_metric.update(logits, pred_probs, data['label'])

    auc = auc_metric.eval()
    net.set_train(True)
    return auc

auc_metric = AUCMetric()

print("✓ quick_auc_eval defined  (direct network inference, no PredictWithSigmoid wrapper)")

In [ ]:
print("="*80)
print(f"STARTING TRAINING   run={RUN_NAME}   device={DEVICE_TARGET}")
print("="*80)
print(f"  Epochs (budget)  : {EPOCHS}  (early stopping may terminate earlier)")
print(f"  Early stop pat.  : 40 epochs of no smoothed-AUC improvement")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Steps / epoch    : {STEPS_PER_EPOCH:,}")
print(f"  Adam LR          : {DEEP_LR}")
print()

BEST_AUC            = 0.0
BEST_SMOOTH_AUC     = 0.0
BEST_CKPT_PATH      = None
PRINT_EVERY         = max(1, STEPS_PER_EPOCH // 3)
SMOOTH_WINDOW       = 5
EARLY_STOP_PATIENCE = 150
no_improve_count    = 0
stopped_early       = False

t_total = time.time()

for epoch in range(1, EPOCHS + 1):
    t_epoch       = time.time()
    loss_sum      = 0.0
    step_count    = 0

    train_net.set_train(True)

    for data in train_dataset.create_dict_iterator():
        # TrainOneStepCell returns a scalar: loss_w + loss_d
        total_loss_t = train_net(data['feat_ids'], data['feat_vals'], data['label'])
        loss_val      = float(total_loss_t.asnumpy())
        loss_sum     += loss_val
        step_count   += 1

        if step_count % PRINT_EVERY == 0:
            elapsed = time.time() - t_epoch
            eta_min = (STEPS_PER_EPOCH - step_count) / max(step_count / elapsed, 1) / 60
            print(f"  step {step_count:>4}/{STEPS_PER_EPOCH:<5}  "
                  f"loss={loss_val:.4f}  ETA {eta_min:.1f}m")

    avg_loss   = loss_sum / step_count
    epoch_time = time.time() - t_epoch

    # ── Per-epoch AUC ─────────────────────────────────────────────────────
    auc = quick_auc_eval(test_dataset, network, auc_metric)

    history['epoch'].append(epoch)
    history['total_loss'].append(avg_loss)
    history['auc'].append(auc)
    history['epoch_time_s'].append(epoch_time)

    # ── Smoothed AUC (trailing mean over last SMOOTH_WINDOW epochs) ───────
    recent_aucs  = [v for v in history['auc'][-SMOOTH_WINDOW:] if v is not None]
    smooth_auc   = float(np.mean(recent_aucs)) if recent_aucs else 0.0

    # ── Checkpoint on single-epoch best ───────────────────────────────────
    if auc is not None and auc > BEST_AUC:
        BEST_AUC       = auc
        BEST_CKPT_PATH = str(RUN_CKPT / 'dao_wide_deep_best.ckpt')
        ms.save_checkpoint(network, BEST_CKPT_PATH)
        print(f"  ★ New best AUC: {BEST_AUC:.4f}  smooth={smooth_auc:.4f}")

    # ── Early stopping on smoothed AUC ────────────────────────────────────
    if smooth_auc > BEST_SMOOTH_AUC + 1e-4:
        BEST_SMOOTH_AUC  = smooth_auc
        no_improve_count = 0
    else:
        no_improve_count += 1

    print(f"  ▶ Epoch {epoch:>3}  loss={avg_loss:.4f}  "
          f"AUC={f'{auc:.4f}' if auc is not None else 'N/A'}  "
          f"smooth={smooth_auc:.4f}  "
          f"no_improve={no_improve_count}/{EARLY_STOP_PATIENCE}  "
          f"t={epoch_time/60:.1f}m")

    if epoch % 5 == 0:
        clear_output(wait=True)
        plot_history(history, save_path=str(log_dir / f'training_curves_{RUN_NAME}.png'))

    if no_improve_count >= EARLY_STOP_PATIENCE:
        print(f"\n  ⏹  Early stopping at epoch {epoch} — "
              f"smoothed AUC flat for {EARLY_STOP_PATIENCE} epochs")
        stopped_early = True
        break

total_time_min = (time.time() - t_total) / 60
clear_output(wait=True)
plot_history(history, save_path=str(log_dir / f'training_curves_{RUN_NAME}.png'))

print("="*80)
print("TRAINING COMPLETE")
print(f"  Total epochs    : {len(history['epoch'])}  {'(early stop)' if stopped_early else '(full budget)'}")
print(f"  Total time      : {total_time_min:.1f} minutes")
print(f"  Best AUC        : {BEST_AUC:.4f}")
print(f"  Best smooth AUC : {BEST_SMOOTH_AUC:.4f}")
print(f"  Best ckpt       : {BEST_CKPT_PATH}")
print("="*80)

---
## Step 6: Save Artefacts

In [ ]:
print("="*80)
print("SAVING TRAINING ARTEFACTS")
print("="*80)

# 1. Final model checkpoint (WideDeepModel weights only)
FINAL_CKPT = ckpt_dir / 'dao_wide_deep_final.ckpt'
ms.save_checkpoint(network, str(FINAL_CKPT))
print(f"\n✓ Final checkpoint : {FINAL_CKPT}")

# 2. Training configuration (everything needed to reproduce this run)
config_out = {
    'run_name'         : RUN_NAME,
    'batch_size'       : config.batch_size,
    'field_size'       : config.field_size,
    'vocab_size'       : config.vocab_size,
    'emb_dim'          : config.emb_dim,
    'deep_layer_dim'   : config.deep_layer_dim,
    'deep_layer_act'   : config.deep_layer_act,
    'l2_coef'          : config.l2_coef,
    'epochs'           : EPOCHS,
    'device_target'    : DEVICE_TARGET,
    'cat_vocab_sizes'  : CAT_VOCAB_SIZES,
    'cat_offsets'      : CAT_OFFSETS.tolist(),
    'cont_base_id'     : int(CONT_BASE_ID),
    'skill_base_id'    : int(SKILL_BASE_ID),
    'sim_skill_base'   : int(SIM_SKILL_BASE),
    'best_auc'         : float(BEST_AUC),
    'best_ckpt_path'   : BEST_CKPT_PATH,
    'total_time_min'   : total_time_min,
}

CKPT_CONFIG = ckpt_dir / 'training_config.json'
with open(CKPT_CONFIG, 'w') as f:
    json.dump(config_out, f, indent=2)
print(f"✓ Training config  : {CKPT_CONFIG}")

# 3. Training history CSV
history_df = pd.DataFrame(history)
HISTORY_CSV = log_dir / f'training_history_{RUN_NAME}.csv'
history_df.to_csv(HISTORY_CSV, index=False)
print(f"✓ History CSV      : {HISTORY_CSV}")

# 4. Final chart
plot_history(history, save_path=str(log_dir / f'training_curves_{RUN_NAME}_final.png'))
print(f"✓ Final chart      : {log_dir / f'training_curves_{RUN_NAME}_final.png'}")

print()
print("Training History Summary:")
display(history_df.round(4))

---
## Step 7: Evaluation

Loads the best checkpoint into a fresh model instance and runs full
inference on the test set. Uses the official `AUCMetric` accumulator,
then adds sklearn metrics for a complete picture.

In [ ]:
print("="*80)
print("LOADING BEST CHECKPOINT FOR EVALUATION")
print("="*80)

# Load best weights back into the SAME 'network' object used during training.
#
# Why not a fresh WideDeepModel? After TrainStepWrap wraps loss_net (which
# wraps network with auto_prefix=True), MindSpore renames network's parameters
# with a 'network.' prefix. The checkpoint is saved with those prefixed names.
# Loading into a fresh WideDeepModel causes 'Wide_b' and
# 'deep_embeddinglookup.embedding_table' to be silently skipped, producing
# a partially-initialised model (observed: AUC 0.44, F1 0.00).
# Loading into the same object works because it already has the renamed params.

ckpt_to_load = BEST_CKPT_PATH if BEST_CKPT_PATH else str(FINAL_CKPT)
param_dict   = ms.load_checkpoint(ckpt_to_load)
ms.load_param_into_net(network, param_dict)
network.set_train(False)

print(f"\n✓ Best checkpoint loaded into existing network: {ckpt_to_load}")
print(f"  All parameters restored — no prefix mismatch warnings expected.")

In [ ]:
print("="*80)
print("COLLECTING PREDICTIONS ON TEST SET")
print("="*80)

sigmoid_op = ops.Sigmoid()
all_probs  = []
all_labels = []
n_done     = 0
t_inf      = time.time()

for data in test_dataset.create_dict_iterator():
    logits, _  = network(data['feat_ids'], data['feat_vals'])
    probs       = sigmoid_op(logits)
    all_probs.append(probs.asnumpy().flatten())
    all_labels.append(data['label'].asnumpy().flatten().astype(int))
    n_done += data['label'].shape[0]
    if n_done % 5000 == 0:
        print(f"  {n_done:>7,} samples processed ...")

all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

print(f"\n✓ Inference complete in {time.time()-t_inf:.1f}s")
print(f"  Test samples : {len(all_probs):,}")
print(f"  Prob range   : [{all_probs.min():.4f}, {all_probs.max():.4f}]")
print(f"  Label dist   : {(all_labels==1).sum():,} pos / {(all_labels==0).sum():,} neg")

In [ ]:
print("="*80)
print("EVALUATION METRICS")
print("="*80)

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, roc_curve, precision_recall_curve, confusion_matrix
)

# Find the threshold that maximises F1 (instead of fixed 0.5)
# This matters when classes are imbalanced — the optimal threshold shifts.
thresholds  = np.linspace(0.1, 0.9, 81)
f1_scores   = [f1_score(all_labels, (all_probs >= t).astype(int), zero_division=0)
               for t in thresholds]
THRESHOLD   = float(thresholds[np.argmax(f1_scores)])
preds_bin   = (all_probs >= THRESHOLD).astype(int)

auc         = roc_auc_score(all_labels, all_probs)
accuracy    = accuracy_score(all_labels, preds_bin) * 100
f1          = f1_score(all_labels, preds_bin, zero_division=0)
precision   = precision_score(all_labels, preds_bin, zero_division=0)
recall      = recall_score(all_labels, preds_bin, zero_division=0)

print(f"\n  {'Metric':<22} {'Value'}")
print(f"  {'─'*38}")
print(f"  {'AUC-ROC':<22} {auc:.4f}")
print(f"  {'Accuracy':<22} {accuracy:.2f}%")
print(f"  {'F1 Score':<22} {f1:.4f}  (at optimal threshold)")
print(f"  {'Precision':<22} {precision:.4f}")
print(f"  {'Recall':<22} {recall:.4f}")
print(f"  {'Threshold (optimal)':<22} {THRESHOLD:.2f}  (maximises F1)")
print(f"  {'Threshold (0.50)':<22} F1={f1_score(all_labels, (all_probs>=0.5).astype(int), zero_division=0):.4f}")


results = {
    'auc_roc'    : float(auc),
    'accuracy'   : float(accuracy),
    'f1_score'   : float(f1),
    'precision'  : float(precision),
    'recall'     : float(recall),
    'threshold'  : THRESHOLD,
    'n_test'     : int(len(all_labels)),
    'checkpoint' : ckpt_to_load,
}

RESULTS_PATH = ckpt_dir / 'evaluation_results.json'
with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n✓ Results saved to {RESULTS_PATH}")

In [ ]:
print("="*80)
print("EVALUATION CHARTS")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[0].plot(fpr, tpr, color='#3498db', linewidth=2,
             label=f'AUC = {auc:.4f}')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall curve
prec_vals, rec_vals, _ = precision_recall_curve(all_labels, all_probs)
axes[1].plot(rec_vals, prec_vals, color='#2ecc71', linewidth=2)
axes[1].fill_between(rec_vals, prec_vals, alpha=0.1, color='#2ecc71')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall  (F1={f1:.3f})')

# Confusion matrix
cm = confusion_matrix(all_labels, preds_bin)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
axes[2].set_title(f'Confusion Matrix  (threshold={THRESHOLD})')

plt.suptitle(
    f'Evaluation — {RUN_NAME}  |  AUC={auc:.4f}  Accuracy={accuracy:.2f}%',
    fontsize=13
)
plt.tight_layout()

chart_path = log_dir / f'evaluation_charts_{RUN_NAME}.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Charts saved to {chart_path}")

In [ ]:
print("="*80)
print("NOTEBOOK 11 COMPLETE")
print("="*80)

print(f"\n  Run name    : {RUN_NAME}")
print(f"  AUC-ROC     : {auc:.4f}")
print(f"  Accuracy    : {accuracy:.2f}%")
print(f"  F1 Score    : {f1:.4f}")
print(f"  Best AUC    : {BEST_AUC:.4f}")
print(f"  Train time  : {total_time_min:.1f} minutes")

print(f"\n  Files saved:")
print(f"    {BEST_CKPT_PATH}")
print(f"    {FINAL_CKPT}")
print(f"    {CKPT_CONFIG}")
print(f"    {HISTORY_CSV}")
print(f"    {RESULTS_PATH}")
print(f"    {log_dir / f'evaluation_charts_{RUN_NAME}.png'}")

print()
if auc is not None and auc >= 0.75:
    print("  ✓ AUC ≥ 0.75 — model meets performance target")
    print("  → Next: Notebook 12 — Hyperparameter tuning")
    print("    Key levers: emb_dim (try 32, 64), deep_layer_dim, epochs")
else:
    print("  ⚠  AUC below 0.75 — recommended next steps:")
    print("     1. Increase emb_dim from 16 to 32 or 64 in CONFIG (Step 1)")
    print("     2. Increase epochs to 15")
    print("     3. Try deep_layer_dim=[512, 256, 128, 64]")
    print("     4. Check class imbalance in Notebook 10 Step 5")
    print("  → Adjust config in Step 1 and re-run from Step 3 onwards")